# 02 — YOLOv8-nano Logo Detector Training
## Brand Vision — Local RTX 4050 or Kaggle P100

**Prerequisite:** Run `01_data_preparation.ipynb` first.

- **Local (RTX 4050):** `batch=16`, ~45–90 min for 50 epochs
- **Kaggle P100:** `batch=32`, ~25–40 min for 60 epochs

Update `DATA_YAML_PATH` below before running.

In [ ]:
# Cell 1 — Install (run on Kaggle; skip locally — already in venv)
# !pip install ultralytics -q

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3 — Config
from pathlib import Path

# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
# Local:  absolute path to your logo_data.yaml
# Kaggle: "/kaggle/working/yolo_logo_detection/logo_data.yaml"
DATA_YAML_PATH = str(
    Path("datasets/yolo_logo_detection/logo_data.yaml").resolve()
)
# ─────────────────────────────────────────────────────────────────────────────

# Detect platform: Kaggle has 16GB P100, local has 6GB RTX 4050
import os
ON_KAGGLE = os.path.exists("/kaggle")
BATCH     = 32 if ON_KAGGLE else 16   # max 16 on local 6GB VRAM
EPOCHS    = 60 if ON_KAGGLE else 50
CACHE     = "ram" if ON_KAGGLE else True

print(f"Running on: {'Kaggle' if ON_KAGGLE else 'local'}")
print(f"data yaml:  {DATA_YAML_PATH}")
print(f"batch={BATCH}, epochs={EPOCHS}")

In [ ]:
# Cell 4 — Train YOLOv8-nano
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Downloads COCO pretrained weights on first run

results = model.train(
    data      = DATA_YAML_PATH,
    epochs    = EPOCHS,
    imgsz     = 640,
    batch     = BATCH,
    device    = 0,             # GPU 0
    project   = "runs/logo_detector",
    name      = "v1",
    pretrained= True,
    patience  = 10,            # Early stopping
    lr0       = 0.01,
    lrf       = 0.001,
    optimizer = "AdamW",
    amp       = True,          # AMP — saves VRAM
    cache     = CACHE,
    workers   = 4,
    save      = True,
    save_period = 10,
    mosaic    = 1.0,
    fliplr    = 0.5,
    degrees   = 5.0,
    warmup_epochs = 3,
    verbose   = True,
)

print(f"\nTraining complete!")
print(f"Best weights: {results.save_dir}/weights/best.pt")

In [ ]:
# Cell 5 — Validate
best_pt = f"{results.save_dir}/weights/best.pt"
model_best = YOLO(best_pt)
metrics = model_best.val(data=DATA_YAML_PATH)

map50 = metrics.box.map50
print(f"mAP50:     {map50:.4f}  (target: >0.60, ideal: >0.75)")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

if map50 >= 0.60:
    print("\n✅ Phase 2 acceptance criteria met! Proceed to ONNX export.")
else:
    print("\n⚠ mAP50 below 0.60 — consider training more epochs or checking dataset.")

In [ ]:
# Cell 6 — Export to ONNX
model_best.export(format="onnx", imgsz=640, dynamic=True, simplify=True)
print(f"ONNX saved to: {results.save_dir}/weights/best.onnx")
print("Next: copy best.onnx to backend/weights/logo_detector.onnx")